# Alchemy GeomML + TDA — Kaggle Run

Полный пайплайн обучения 6 моделей на Alchemy датасете через Kaggle GPU.

**ВАЖНО:** Запускай через **Save Version (Commit)**, не через интерактивную сессию!
Иначе output не сохранится.

**Датасет:** Alchemy v20191129 (202,579 молекул)

**Модели:** FCNN, SchNet, EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA

**GPU:** Kaggle T4 (16 ГБ) или P100 (16 ГБ) — бесплатно 30ч/неделю

## 1. Подготовка окружения (2 минуты)

Клонируем репозиторий и проверяем, что GPU доступен.

In [ ]:
%cd /kaggle/working
!rm -rf alchemy-geom-tda
!git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
%cd alchemy-geom-tda

# Проверка версии и актуальных фич
!git log --oneline -1
!echo "=== Проверка версии ==="
!echo "PyGDataParallel: $(grep -c 'PyGDataParallel' src/train.py)"
!echo "DataListLoader:  $(grep -c 'DataListLoader' src/train.py)"
!echo "multi_gpu:       $(grep -c 'multi_gpu' src/train.py)"
!echo "num_workers:     $(grep -c 'num_workers' src/train.py)"
!echo "★ best:          $(grep -c '★ best' src/train.py)"

## 2. Установка зависимостей (3 минуты)

Kaggle уже имеет torch + numpy + pandas предустановленными. Ставим только недостающие.

In [ ]:
!pip install torch-geometric gudhi rdkit egnn-pytorch -q

# Проверка
import torch
import torch_geometric
import egnn_pytorch
import rdkit
import gudhi

print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU count: {torch.cuda.device_count()}')
print(f'torch-geometric: {torch_geometric.__version__}')
print(f'rdkit: {rdkit.__version__}')
print(f'gudhi: {gudhi.__version__}')

## 3. Скачивание датасета Alchemy (2 минуты)

136 МБ zip, ~600 МБ после распаковки. SHA256 проверка автоматически.

In [ ]:
!python data/download_alchemy.py 2>&1 | tail -15

## 4. Smoke-тест (5 минут, проверка что всё работает)

Обучаем EGNN на 100 молекулах, 2 эпохи. Должен быть виден убывающий loss.

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src')
os.makedirs('results/smoke', exist_ok=True)

sys.argv = ['train.py',
    '--model', 'egnn',
    '--target', 'all',
    '--epochs', '2',
    '--max_train', '100',
    '--max_val', '20',
    '--max_test', '20',
    '--batch_size', '32',
    '--device', 'cuda',
    '--output_dir', 'results/smoke',
    '--num_workers', '2',
]

from train import main as train_main
train_main()

## 5. Полное обучение — все 6 моделей (4-8 часов на T4 GPU)

Запуск через `run_all.py` — последовательно обучает FCNN, SchNet, EGNN, EGNN+TDA, EGNN Vector, EGNN Vector+TDA.

Параметры:
- `--epochs 100` — максимум (EarlyStopping остановит раньше)
- `--batch_size 512` — для T4 16GB
- `--multi_gpu` — если доступно 2+ GPU
- `--num_workers 4` — для ускорения DataLoader
- `--es_mode or` — быстрее останавливается (когда ХОТЯ БЫ ОДНА метрика перестала улучшаться)

In [ ]:
import os
import sys
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src')

# Раскомментируй для запуска всех 6 моделей (4-8 часов на T4 GPU)
# ⚠️ ВСЕГДА запускай через Save Version (Commit), не через интерактив!

sys.argv = ['run_all.py',
    '--epochs', '100',
    '--batch_size', '512',
    '--hidden_channels', '128',
    '--num_layers', '4',
    '--lr', '1e-3',
    '--lr_patience', '5',
    '--device', 'cuda',
    '--patience', '15',
    '--models', 'all',  # или 'egnn,egnn_tda,egnn_vector,egnn_vector_tda'
    '--num_workers', '4',
    '--multi_gpu',  # раскомментируй если 2+ GPU
]

# from run_all import main as run_all_main
# run_all_main()

## 6. Альтернатива: только 4 EGNN-модели (быстрее, ~3-5 часов)

FCNN и SchNet можно обучить отдельно — они быстрее.

In [ ]:
# Раскомментируй для запуска только EGNN-семьи

# sys.argv = ['run_all.py',
#     '--epochs', '100',
#     '--batch_size', '512',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--lr_patience', '5',
#     '--device', 'cuda',
#     '--patience', '15',
#     '--models', 'egnn,egnn_tda,egnn_vector,egnn_vector_tda',
#     '--num_workers', '4',
#     '--multi_gpu',
# ]
# from run_all import main as run_all_main
# run_all_main()

## 7. Альтернатива: только одна модель (быстрая проверка, ~1 час)

Обучить только EGNN на полном датасете, 50 эпох.

In [ ]:
# Раскомментируй для запуска только EGNN

# sys.argv = ['train.py',
#     '--model', 'egnn',
#     '--target', 'all',
#     '--epochs', '50',
#     '--batch_size', '512',
#     '--hidden_channels', '128',
#     '--num_layers', '4',
#     '--lr', '1e-3',
#     '--device', 'cuda',
#     '--num_workers', '4',
#     '--output_dir', 'results/experiments/batch_size_512',
# ]
# from train import main as train_main
# train_main()

## 8. Построение графиков обучения (1 минута)

Запускать после завершения обучения. Графики сохранятся в `results/figures/`.

In [ ]:
# После завершения обучения

# sys.argv = ['plot.py',
#     '--input_dir', 'results/experiments/batch_size_512',
#     '--save_dir', 'results/figures/batch_size_512',
#     '--no-show',
# ]
# from plot import plot_main
# plot_main()

# Или отдельно parity plots
# import glob
# from plot import plot_parity
# for csv in glob.glob('results/experiments/batch_size_512/history_*.csv'):
#     model_name = csv.split('/')[-1].replace('history_', '').rsplit('_', 2)[0]
#     plot_parity(csv, save_path=f'results/figures/batch_size_512/{model_name}_parity.png', show=False)

## 9. Robustness evaluation (30 минут на GPU)

Оценка устойчивости обученной модели к шуму в координатах.

Запускать после завершения обучения для каждой модели отдельно.

In [ ]:
# Раскомментируй для robustness eval на EGNN

# sys.argv = ['eval_robustness.py',
#     '--model', 'egnn', '--target', 'all',
#     '--checkpoint', 'checkpoints/egnn_all_best.pt',
#     '--target_stats', 'results/experiments/batch_size_512/target_stats_egnn_all.json',
#     '--noise_sigma', '0.0,0.05,0.10,0.15',
#     '--device', 'cuda',
# ]
# from eval_robustness import main as eval_main
# eval_main()

## 10. AutoML — автоматический выбор архитектуры (5 минут)

Программа максимум: извлечение геометрических prior-ов из TDA и рекомендация архитектуры.

In [ ]:
# AutoML — извлечение priors из TDA
sys.path.insert(0, '/kaggle/working/alchemy-geom-tda/src/automl')
sys.argv = ['select.py',
    '--data_dir', 'data/alchemy',
    '--n_molecules', '100',
    '--threshold', '0.95',
    '--output_json', 'results/automl/recommendation.json',
    '--device', 'cpu',  # AutoML не требует GPU
]

from select import main as automl_main
automl_main()

## 11. Просмотр отчёта AutoML

In [ ]:
import json
from pathlib import Path

report_path = 'results/automl/recommendation.json'
if Path(report_path).exists():
    with open(report_path) as f:
        report = json.load(f)
    print('=== AutoML Report ===')
    print(f'Timestamp: {report["timestamp"]}')
    print(f'\nPriors (n={report["priors"]["n_molecules_success"]} molecules):')
    for k, v in report['priors'].items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
    print(f'\nRecommendation:')
    for k, v in report['recommendation'].items():
        print(f'  {k}: {v}')
else:
    print(f'Отчёт не найден: {report_path}')

## 12. Просмотр результатов обучения

In [ ]:
import pandas as pd
import glob

# Найти summary CSV
summary_csvs = sorted(glob.glob('results/experiments/batch_size_*/summary_*.csv'))
if summary_csvs:
    latest_summary = summary_csvs[-1]
    print(f'Latest summary: {latest_summary}')
    df = pd.read_csv(latest_summary)
    print(df.to_string(index=False))
else:
    print('Summary CSV не найден. Сначала запусти полное обучение (ячейка 5).')

# Показать последние несколько строк истории каждой модели
print('\n=== Last 3 epochs of each model ===')
for csv in sorted(glob.glob('results/experiments/batch_size_*/history_*.csv')):
    model_name = csv.split('/')[-1].replace('history_', '').rsplit('_', 2)[0]
    df = pd.read_csv(csv)
    print(f'\n{model_name} ({len(df)} epochs):')
    print(df.tail(3).to_string(index=False))

## 13. ⚠️ СОХРАНИТЬ ЧЕРЕЗ COMMIT

**НЕ запускай последнюю ячейку с обучением в interactive режиме!** Вместо этого:

1. Нажми **Save Version** в правом верхнем углу
2. Выбери **Save & Run All (Commit)**
3. Дай название версии (например: `v32 egnn+tda bs512`)
4. Нажми **Save**

Kaggle запустит ноутбук **в фоновом режиме** с нуля. После завершения:
- Output сохранится навсегда
- Появится в **Versions** tab
- Можно скачать через `kaggle kernels output` или через браузер

## 14. Что сохранится после Commit

В Output будут:
- `checkpoints/<model>_all_best.pt` — чекпойнт каждой модели
- `results/experiments/batch_size_512/history_<model>_all_<ts>.csv` — история
- `results/experiments/batch_size_512/summary_<ts>.csv` — сводка
- `results/experiments/batch_size_512/args_<model>_all.json` — гиперпараметры
- `results/experiments/batch_size_512/target_stats_<model>_all.json` — нормализация
- `results/figures/batch_size_512/*.png` — графики
- `results/robustness/<model>_robustness.csv` — robustness
- `results/automl/recommendation.json` — отчёт AutoML
- `data/alchemy/processed/*.pt` — кэш TDA (для повторных запусков)

## 15. Частые проблемы

### OOM (CUDA out of memory)
Уменьши `--batch_size` до 256:
```python
'--batch_size', '256',
```

### DataParallel device mismatch
Раскомментируй `os.environ['CUDA_VISIBLE_DEVICES'] = '0'` в начале ноутбука.
Или используй `--device cuda:0` (v32+) для выбора конкретной GPU.

### Не нужны все 4 модели
Сократи список:
```python
'--models', 'egnn,egnn_tda',  # только 2 модели
```

### Долгое обучение (v32+)
Если Early Stopping занимает слишком много эпох:
- `--es_mode or` — стоп, когда ХОТЯ БЫ одна метрика перестала улучшаться
- `--es_mode loss_only` — только val_loss (классический ES)
- `--patience 10` (вместо 15) — раньше останавливаться

### TDA ухудшает метрики (v32+)
Если `egnn_tda` работает хуже `egnn`:
- `--tda_mode film` — FiLM-модуляция вместо concat
- TDA-фичи нормализуются через BatchNorm1d, но film может быть стабильнее

### 12h лимит истечёт
При Commit Kaggle даёт 12h на выполнение. Если не успел:
- Все файлы созданные до таймаута сохранятся
- Но если `torch.save()` не успел вызваться — чекпойнта не будет
- `torch.save()` вызывается один раз в конце — если нужно periodic save, можно модифицировать train.py

## 16. Скачать результаты после Commit

После завершения Commit:

```bash
# Через kaggle CLI
pip install kaggle
kaggle kernels output <username>/<notebook-slug> -p ~/alchemy_output

# Или через браузер
# Kaggle → notebook → Output tab → Download
```

## 17. Что делать с результатами

1. **Загрузить чекпойнты и CSV на GitHub** (через Git LFS — см. `.gitattributes`)
2. **Обновить `results/table.md`** финальными метриками
3. **Прогнать `eval_robustness.py`** на каждой модели для robustness таблицы
4. **Опубликовать v32 release** на GitHub с тегом `v32.38`